In [ ]:
!pip install yfinance xgboost lightgbm requests

import pandas as pd
import numpy as np
import yfinance as yf
import requests

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import pickle

In [ ]:
stocks = ["NTPC.NS", "TATAPOWER.NS", "ADANIPOWER.NS", "JSWENERGY.NS", "NHPC.NS"]

company_type_map = {
    "NTPC.NS": 0,
    "ADANIPOWER.NS": 0,
    "TATAPOWER.NS": 1,
    "JSWENERGY.NS": 1,
    "NHPC.NS": 2
}

In [ ]:
data = yf.download(stocks, start="2016-01-01")['Close']
data = data.resample('W').last().reset_index()

/tmp/ipykernel_9103/1987217485.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(stocks, start="2016-01-01")['Close']
[*********************100%***********************]  5 of 5 completed


In [ ]:
def get_temperature_data():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 20.2961,
        "longitude": 85.8245,
        "start_date": "2016-01-01",
        "end_date": "2024-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    data = response.json()

    temp_df = pd.DataFrame({
        "Date": pd.to_datetime(data["daily"]["time"]),
        "Temp": data["daily"]["temperature_2m_mean"]
    })
    return temp_df

temp_daily = get_temperature_data()
temp = temp_daily.set_index('Date').resample('W').mean().reset_index()

In [ ]:
nifty = yf.download("^NSEI", start="2016-01-01")['Close']
nifty = nifty.resample('W').last().reset_index()
nifty.rename(columns={'Close': 'Nifty'}, inplace=True)

/tmp/ipykernel_9103/3493418676.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI", start="2016-01-01")['Close']
[*********************100%***********************]  1 of 1 completed


In [ ]:
def create_features(df, company_type):

    df['return'] = df['Close'].pct_change()

    # Lag returns (IMPORTANT)
    df['return_lag1'] = df['return'].shift(1)
    df['return_lag2'] = df['return'].shift(2)

    # Climate
    df['temp_lag1'] = df['Temp'].shift(1)
    df['temp_lag2'] = df['Temp'].shift(2)
    df['heatwave'] = (df['Temp'] > 35).astype(int)
    df['temp_change'] = df['Temp'].pct_change()

    # Company-specific climate
    if company_type == 0:
        df['climate_impact'] = df['Temp']
    elif company_type == 1:
        df['climate_impact'] = df['Temp'] * 0.5
    else:
        df['climate_impact'] = -df['Temp']

    # Demand
    df['demand'] = df['Temp'] * (1 + df['heatwave'])

    # Market
    df['nifty_return'] = df['Nifty'].pct_change()

    # Interaction (VERY IMPORTANT)
    df['temp_x_nifty'] = df['Temp'] * df['nifty_return']

    # Financial
    df['ma_3'] = df['Close'].rolling(3).mean()
    df['volatility'] = df['return'].rolling(3).std()
    df['momentum'] = df['Close'] - df['Close'].shift(3)

    df['company_type'] = company_type

    # 🎯 BETTER TARGET (LESS NOISE)
    df['target'] = (df['return'].shift(-1) > 0.02).astype(int)

    df.dropna(inplace=True)

    # 🔥 Remove low-volatility noise
    df = df[df['volatility'] > df['volatility'].quantile(0.3)]

    return df

In [ ]:
combined_dfs_list = []

for stock in stocks:

    df = pd.merge(data[['Date', stock]], temp, on='Date', how='inner')
    df = pd.merge(df, nifty, on='Date', how='inner')

    # Rename the Nifty column from '^NSEI' to 'Nifty'
    df = df.rename(columns={'^NSEI': 'Nifty'})

    df = df.rename(columns={stock: 'Close'})

    df = create_features(df, company_type_map[stock])
    df['company'] = stock

    combined_dfs_list.append(df)

combined_df = pd.concat(combined_dfs_list, ignore_index=True)

In [ ]:
features = [
    'return_lag1','return_lag2',
    'temp_lag1','temp_lag2',
    'heatwave','temp_change',
    'climate_impact','demand',
    'nifty_return','temp_x_nifty',
    'ma_3','volatility','momentum',
    'company_type'
]

X = combined_df[features]
y = combined_df['target']

print("Dataset:", X.shape)

Dataset: (1635, 14)


In [ ]:
scale_pos_weight = len(y[y==0]) / len(y[y==1])

In [ ]:
models = {
    "rf": RandomForestClassifier(n_estimators=200),
    "xgb": XGBClassifier(n_estimators=200, max_depth=4, scale_pos_weight=scale_pos_weight),
    "lgbm": LGBMClassifier(n_estimators=200),
    "lr": LogisticRegression(max_iter=1000)
}

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

model_scores = {}

for name, model in models.items():

    scores = []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        scores.append(accuracy_score(y_test, preds))

    model_scores[name] = np.mean(scores)

print("Model Scores:", model_scores)

[LightGBM] [Info] Number of positive: 75, number of negative: 200
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000090 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1088
[LightGBM] [Info] Number of data points in the train set: 275, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.272727 -> initscore=-0.980829
[LightGBM] [Info] Start training from score -0.980829
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [ ]:
top_models = sorted(model_scores, key=model_scores.get, reverse=True)[:3]
print("Top Models:", top_models)

Top Models: ['lr', 'rf', 'xgb']


In [ ]:
estimators = [(name, models[name]) for name in top_models]

ensemble = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

ensemble.fit(X, y)

StackingClassifier(estimators=[('lr', LogisticRegression(max_iter=1000)),
                               ('rf', RandomForestClassifier(n_estimators=200)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric=None,
                                              feature_types=Non...
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=None, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=4,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=200, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression())

In [ ]:
pickle.dump(ensemble, open("model.pkl", "wb"))
pickle.dump(features, open("features.pkl", "wb"))

print("Final Model Saved!")

Final Model Saved!


In [ ]:
def predict_signal(row):

    prob = ensemble.predict_proba([row])[0][1]

    if prob > 0.65:
        return prob, "BUY"
    elif prob < 0.35:
        return prob, "SELL"
    else:
        return prob, "HOLD"

In [ ]:
def explain(row):

    if row['heatwave'] == 1:
        return "Heatwave → demand surge → thermal stocks benefit"

    elif row['climate_impact'] < 0:
        return "Hydro dependency → affected negatively"

    else:
        return "Moderate climate impact"

In [ ]:
sample = X.iloc[-1]

prob, signal = predict_signal(sample)

print("Probability:", round(prob,2))
print("Signal:", signal)
print("Reason:", explain(sample))

Probability: 0.24
Signal: SELL
Reason: Hydro dependency → affected negatively


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
